# Troiani Tokenizer &mdash; Vocabulary Discovery

**Goal:** Find the optimal BPE vocabulary (50,032 tokens) for a sub-1B hybrid Mamba/GQA LLM.

**Why 50,032?** 50,000 regular tokens + 32 special tokens (`<pad>`, `<bos>`, `<eos>`, `<unk>`, `<mask>`, plus 27 reserved for future use).

**Guiding question:** What dataset mix produces a tokenizer with best compression ratio, coverage, and generalization for our target domain (general-purpose text + code)?

---
## 1. Data Sources &mdash; Candidate Corpora

A tokenizer is only as good as its training data. We need a representative sample of the data the model will actually see during training.

### Recommended sources (ordered by priority)

| Source | Why | Size suggestion | Access |
|---|---|---|---|
| **FineWeb-Edu** | High-quality educational web text | 1&ndash;5 GB sample | Hugging Face: `HuggingFaceFW/fineweb-edu` |
| **The Pile** | Broad domain diversity (academic, books, web, code) | 1&ndash;2 GB sample | EleutherAI: `the_pile` |
| **Wikipedia** | Clean, well-formed prose | 500 MB&ndash;1 GB | Hugging Face: `wikipedia` |
| **Code (The Stack)** | Programming languages for code gen | 500 MB&ndash;1 GB | Hugging Face: `bigcode/the-stack-dedup` |
| **SlimPajama** | Deduplicated, already mixed | 1&ndash;3 GB sample | Hugging Face: `cerebras/SlimPajama-627B` |

### Heuristics for a sub-1B model
- **Small models** benefit from **cleaner, more formal text** &mdash; less noise, better token efficiency per param
- **Code is cheap** (~1.2 tokens/byte for English vs ~2.5+ for code) &mdash; include enough for reasonable coding ability
- **Avoid over-emphasis on a single domain** &mdash; the vocab should be general-purpose

### Suggested mix (experimental starting point)

| Source | Fraction | Rationale |
|---|---|---|
| FineWeb-Edu | 40% | High-quality general text |
| Wikipedia + Books | 20% | Clean formal prose |
| Code (The Stack) | 20% | Programming literacy |
| The Pile (academic/other) | 20% | Domain diversity |

> **Important:** The tokenizer should be trained on raw text, not pre-tokenized data. Use the raw text splits from each dataset.

In [ ]:
# === YOUR CODE: Download and sample datasets ===
# Use datasets.load_dataset with streaming + take() to get samples
# Save samples as .txt files (one line per document or sentence)
# Target: ~1-10 GB of raw text total
from datasets import load_dataset

# Example: FineWeb-Edu sample
# ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
# for i, example in enumerate(ds):
#     if i >= 10000: break
#     # save example["text"] to file
pass

---
## 2. Train Candidate Tokenizers

### Training procedure
1. Concatenate all sampled data into a single text file (or pass multiple files)
2. Use `tokenizers` library (HuggingFace) &mdash; BPE with ByteLevel pre-tokenizer, NFKC normalization (already implemented in `troiani.tokenizer`)
3. Train at multiple vocab sizes to compare:
   - **32,000** &mdash; baseline, known-good size
   - **50,032** &mdash; our target
   - **54,000** &mdash; upper bound tested in earlier sweeps

### What to vary
- **Data mix ratios** (change the proportions above)
- **Min frequency** (default 2, try 3&ndash;5 for cleaner vocabs)
- **Special tokens** (keep consistent: pad=0, bos=1, eos=2, unk=3, mask=4)

In [ ]:
# === YOUR CODE: Train tokenizers ===
from troiani.tokenizer import TroianiTokenizer

# Train at target vocab
t = TroianiTokenizer(vocab_size=50032)
# t.train(files=["data/sample_fineweb.txt", "data/sample_wiki.txt", ...],
#         output_dir="resources/tokenizer")

# Train baseline for comparison
# t32 = TroianiTokenizer(vocab_size=32000)
# t32.train(...)
pass

---
## 3. Evaluation Metrics

### Primary: Compression Ratio

How many tokens does it take to encode a fixed corpus? Lower = better.

$$
\text{Compression} = \frac{\text{bytes}}{\text{tokens}}
$$

| Language | Typical bytes/token |
|---|---|
| English | 3.5&ndash;4.5 |
| Code | 1.5&ndash;2.5 |
| Multilingual | 1.5&ndash;3.0 |

### Secondary: Coverage & Specialization

| Metric | What it measures | How to test |
|---|---|---|
| **Unicode coverage** | Does it handle non-ASCII well? | Encode Chinese, Arabic, emoji |
| **Digit preservation** | Are numbers tokenized well? | Encode years, prices, math |
| **Whitespace handling** | Does ByteLevel add noise? | Encode sentences with/without spaces |
| **OOV rate** | What fraction is `<unk>`? | Should be 0% for BPE |
| **Code tokenization** | Are common keywords single tokens? | Encode `def`, `import`, `if __name__` |

### Ablation: Data Mix Sensitivity

Train tokenizers on different data mixes and compare compression on a **held-out evaluation set** (a fixed corpus not used in any tokenizer training). This tells you which mix produces the most general vocabulary.

In [ ]:
# === YOUR CODE: Evaluate tokenizers ===
from troiani.tokenizer import TroianiTokenizer

# Load trained tokenizer
t = TroianiTokenizer(vocab_size=50032)
# t = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")

# Test on a held-out corpus
# with open("data/holdout.txt") as f:
#     text = f.read()
# encoded = t.encode(text)
# compression_ratio = len(text.encode("utf-8")) / len(encoded)

# Compare tokenizers
# for vs in [32000, 50032, 54000]:
#     t = TroianiTokenizer.from_pretrained(f"resources/tokenizer/tokenizer_{vs}.json")
#     ...
pass

---
## 4. Analysis & Decision

### Decision criteria (in order)
1. **No OOVs** &mdash; every byte should map to some token (BPE property; verify)
2. **Best overall compression** on the held-out set &mdash; lower tokens/byte = more efficient training
3. **Balanced domain compression** &mdash; no extreme bias toward one domain (e.g., great on code but bad on prose)
4. **Reasonable vocab size** &mdash; 50,032 is the target, but if 32K compresses nearly as well, the model saves ~18M embedding params

### Trade-off: Vocabulary size vs Model capacity

At d_model=1088:

| Vocab | Embedding params | Remaining for backbone | Equivalent layers |
|---|---|---|---|
| 32,000 | 34.8M | 886M | ~35 layers |
| **50,032** | **54.4M** | **866M** | **~34 layers** |
| 54,000 | 58.8M | 862M | ~34 layers |

> Bigger vocab = better tokenization but fewer params for the backbone. If 50K compresses >=15% better than 32K, it's worth the trade-off.

### Final deliverable
1. A `resources/tokenizer/tokenizer_50032.json` file (the trained tokenizer)
2. A `resources/tokenizer/tokenizer_50032_config.json` with metadata
3. A decision log (this notebook) showing why 50,032 was chosen over alternatives

In [ ]:
# === YOUR CODE: Finalize and save ===
from troiani.tokenizer import TroianiTokenizer

# Train final tokenizer on the best data mix
t = TroianiTokenizer(vocab_size=50032)
# t.train(files=[...], output_dir="resources/tokenizer")

# Verify
# t2 = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")
# print(f"Vocab size: {len(t2)}")
# print(f"Compression: {...}")
pass

---
## 5. Sanity Checks Before Production

1. **Load tokenizer** &rarr; encode text &rarr; decode &rarr; verify it round-trips
2. **Check special token IDs** are correct (pad=0, unk=3, etc.)
3. **No `None` tokens** in the vocabulary file
4. **Test on real data**: encode a batch of training samples, check for anomalies
5. **Profile speed**: the tokenizer should process >=100K tokens/sec

In [ ]:
from troiani.tokenizer import TroianiTokenizer
tok = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")
assert tok.decode(tok.encode("Hello, world!")) == "Hello, world!"
print("Round-trip: OK")
print(f"Final vocab size: {len(tok)}")

---
## Appendix: Quick Reference

### TroianiTokenizer API

```python
# Training
t = TroianiTokenizer(vocab_size=50032)
t.train(files=["data/corpus.txt"], output_dir="resources/tokenizer")

# Inference
t = TroianiTokenizer.from_pretrained("resources/tokenizer/tokenizer_50032.json")
ids = t.encode("Your text here")
text = t.decode(ids)
```

### Recommended reading
- [HuggingFace tokenizers docs](https://huggingface.co/docs/tokenizers/index)
- [BPE original paper (Sennrich et al.)](https://arxiv.org/abs/1508.07909)
- [FineWeb dataset](https://huggingface.co/datasets/HuggingFaceFW/fineweb)
- [The Pile](https://pile.eleuther.ai/)